In [1]:
import torch

## Step 1 - Verify PyTorch and GPU

Before building the model, verify that the Python environment can import PyTorch and that CUDA is available from the same notebook kernel.

The important check is `torch.cuda.is_available()`. If it prints `True`, tensors and models can be moved to `cuda`. The GPU name check confirms which GPU PyTorch is using.

The model code later uses this pattern:

```python
device = "cuda" if torch.cuda.is_available() else "cpu"
```

That keeps the notebook runnable on CPU, while using the GPU when it is available.


In [2]:
print(torch.__version__)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'no cuda')

2.13.0+cu126
True
NVIDIA GeForce GTX 1080 Ti


## Step 2 - Define the Tiny Vocabulary

This notebook uses a character-level vocabulary with exactly three tokens:

```text
a, b, c
```

`stoi` means string-to-integer. It maps characters to token IDs:

```text
a -> 0
b -> 1
c -> 2
```

`itos` means integer-to-string. It maps token IDs back to characters:

```text
0 -> a
1 -> b
2 -> c
```

`encode()` turns text into token IDs. `decode()` turns token IDs back into text. The basic correctness check is:

```python
decode(encode("abacaba")) == "abacaba"
```

Here `vocab_size = 3` because the model has three possible next-token classes.


In [3]:
stoi_dict = {"a": 0, "b": 1, "c": 2}
itos_dict = {0: "a", 1: "b", 2: "c"}

def stoi(s: str) -> int:
    return stoi_dict[s]

def itos(i: int) -> str:
    return itos_dict[i]

def encode(word: str) -> list[int]:
    return [stoi(s) for s in word]

def decode(nums: list[int]) -> str:
    return ''.join(itos(i) for i in nums)

vocab = ['a', 'b', 'c']
vocab_size = len(vocab)

In [4]:
enc = encode("abca")
dec = decode(enc)

print(enc, dec)

[0, 1, 2, 0] abca


## Step 3 - Create a Tiny Training Corpus

The training corpus is a fixed string containing only `a`, `b`, and `c`:

```python
text = "abacabaacbbccabacabaacbbcc"
```

The model does not train directly on characters. It trains on integer token IDs, so the text is encoded and stored as a one-dimensional PyTorch tensor:

```python
data = torch.tensor(encode(text), dtype=torch.long)
```

`torch.long` is used because token IDs are integer class/index values. Embedding layers and cross-entropy targets expect integer token IDs, not floating point vectors.

Useful checks:

```text
data.ndim == 1
data.min() == 0
data.max() == 2
decode(data.tolist()) == text
```


In [5]:
text = "abacabaacbbccabacabaacbbcc"
data = torch.tensor(encode(text), dtype=torch.long)
data

tensor([0, 1, 0, 2, 0, 1, 0, 0, 2, 1, 1, 2, 2, 0, 1, 0, 2, 0, 1, 0, 0, 2, 1, 1,
        2, 2])

In [6]:
print(text)
print(encode(text))
print(data)
print(data.shape)
print(data.dtype)
print(data.min().item(), data.max().item())
print(decode(data.tolist()))

abacabaacbbccabacabaacbbcc
[0, 1, 0, 2, 0, 1, 0, 0, 2, 1, 1, 2, 2, 0, 1, 0, 2, 0, 1, 0, 0, 2, 1, 1, 2, 2]
tensor([0, 1, 0, 2, 0, 1, 0, 0, 2, 1, 1, 2, 2, 0, 1, 0, 2, 0, 1, 0, 0, 2, 1, 1,
        2, 2])
torch.Size([26])
torch.int64
0 2
abacabaacbbccabacabaacbbcc


In [7]:
decode(data.tolist()) == text

True

## Step 4 - Build Context-Target Examples

For the first baseline, each training example is:

```text
previous block_size tokens -> one next token
```

With `block_size = 4`:

```text
context: a b a c
target:          a
```

Numerically:

```text
x = [0, 1, 0, 2]
y = 0
```

This `block_size` is the toy version of a GPT context window. It controls how many previous token positions the model can see. It is different from `vocab_size`:

```text
vocab_size = how many token types exist = 3
block_size = how many token positions are visible = 4 here
```

`get_batch()` samples random start positions from `data`, then stacks the resulting examples:

```text
xb.shape == [batch_size, block_size]
yb.shape == [batch_size]
```

For this first baseline, `yb` has one target token per context.


In [8]:
block_size = 4
batch_size = 8 

def get_batch():
    ix = torch.randint(0, len(data) - block_size, (batch_size,))
    xb = torch.stack([data[i:i + block_size] for i in ix])
    yb = torch.stack([data[i + block_size] for i in ix])
    return xb, yb

xb, yb = get_batch()

print(xb.shape)  # should be [batch_size, block_size]
print(yb.shape)  # should be [batch_size]

for i in range(batch_size):
    context = decode(xb[i].tolist())
    target = decode([yb[i].item()])
    print(context, "->", target)

torch.Size([8, 4])
torch.Size([8])
baca -> b
caba -> c
ccab -> a
cbbc -> c
acab -> a
acbb -> c
abac -> a
abaa -> c


## Step 5 - Train a Simple Baseline Model

Before building GPT-style attention, this baseline learns:

```text
given 4 previous tokens -> predict the next token
```

The model path is:

```text
token IDs -> token embeddings -> flattened context -> linear layer -> logits
```

`n_embd = 8` is an arbitrary small embedding size. It means each token ID becomes 8 learned numbers. With `vocab_size = 3`, the embedding table has:

```text
3 * 8 = 24 learned parameters
```

For a batch shaped `[8, 4]`, the embedding layer produces:

```text
[batch_size, block_size] -> [8, 4, 8]
```

Flattening turns each context into one vector:

```text
[8, 4, 8] -> [8, 32]
```

The final linear layer maps each 32-number context vector to 3 raw scores:

```text
[8, 32] -> [8, 3]
```

Those raw scores are logits for `a`, `b`, and `c`. `F.cross_entropy(logits, targets)` compares the logits with the correct next-token IDs. Do not apply softmax before `cross_entropy`; the loss function expects raw logits.

The training loop repeats this sequence:

```text
sample batch -> forward pass -> compute loss -> clear gradients -> backward pass -> optimizer step
```


In [9]:
import torch.nn as nn
import torch.nn.functional as F

n_embd = 8
device = "cuda" if torch.cuda.is_available() else "cpu"

class ContextModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, n_embd)
        self.lm_head = nn.Linear(block_size * n_embd, vocab_size)

    def forward(self, idx, targets=None):
        x = self.token_embedding(idx)   # [B, T, C]
        B, T, C = x.shape
        x = x.reshape(B, T * C)         # [B, T*C]
        logits = self.lm_head(x)        # [B, vocab_size]

        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits, targets)

        return logits, loss

In [10]:
model = ContextModel().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-2)

for step in range(1000):
    xb, yb = get_batch()
    xb = xb.to(device)
    yb = yb.to(device)

    logits, loss = model(xb, yb)

    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

    if step % 100 == 0:
        print(step, loss.item())

0 1.7028824090957642
100 0.1866471767425537
200 0.4299897849559784
300 0.06891822814941406
400 0.10819018632173538
500 0.37021684646606445
600 0.37076932191848755
700 0.4382733404636383
800 0.02506132423877716
900 0.004914580844342709


## Step 6 - Generate Text from the Baseline

Generation repeatedly predicts one token and appends it to the sequence.

At each step, the model receives only the last `block_size` tokens:

```python
idx_cond = idx[-block_size:]
```

Then the shape is changed from one sequence to one batch of one sequence:

```text
[T] -> [1, T]
```

The model returns logits for the next token. Softmax converts those logits into probabilities over the three possible tokens:

```text
probability of a
probability of b
probability of c
```

`torch.multinomial()` samples from those probabilities. The sampled token is appended to the running sequence, and the process repeats.

`@torch.no_grad()` disables gradient tracking during generation, because generation is inference, not training. `model.eval()` switches the model into evaluation mode, and `model.train()` switches it back afterward.


In [11]:
@torch.no_grad()
def generate(model, start, max_new_tokens):
    model.eval()

    idx = torch.tensor(encode(start), dtype=torch.long, device=device)

    for _ in range(max_new_tokens):
        idx_cond = idx[-block_size:]          # last block_size tokens
        idx_cond = idx_cond.unsqueeze(0)      # [T] -> [1, T]

        logits, loss = model(idx_cond)
        probs = F.softmax(logits, dim=-1)     # [1, vocab_size]

        idx_next = torch.multinomial(probs, num_samples=1)  # [1, 1]
        idx = torch.cat((idx, idx_next.squeeze(0)), dim=0)

    model.train()
    return decode(idx.tolist())

In [12]:
print(generate(model, "abac", 20))

abacabaacbbccabaacbbccab


## Step 7a - Move Toward GPT-Style Training

The baseline predicted only one token after the whole context. GPT-style training predicts the next token at every position inside the block.

The batch target changes from:

```text
yb.shape == [B]
```

to:

```text
yb.shape == [B, T]
```

Example:

```text
xb: a b a c
yb: b a c a
```

So the model learns all of these predictions from one row:

```text
a       -> b
a b     -> a
a b a   -> c
a b a c -> a
```

The MiniGPT skeleton adds two embeddings:

```text
token embedding:    what token is this?
position embedding: where is this token inside the context window?
```

For input `a b c a`, the two `a` tokens have the same token embedding, but different positional embeddings:

```text
first a  = token_embedding[a] + position_embedding[0]
second a = token_embedding[a] + position_embedding[3]
```

In tensor form:

```python
x = tok_emb + pos_emb
```

with shapes:

```text
tok_emb: [B, T, C]
pos_emb:    [T, C]
result:  [B, T, C]
```

PyTorch broadcasts `pos_emb` across the batch dimension. Addition keeps the hidden size equal to `n_embd`; concatenation would double it and require changing later layers.

The logits now have shape:

```text
[B, T, vocab_size]
```

For cross-entropy, the batch and time dimensions are flattened:

```text
logits:  [B, T, vocab_size] -> [B*T, vocab_size]
targets: [B, T]             -> [B*T]
```


In [13]:
def get_batch():
    ix = torch.randint(0, len(data) - block_size, (batch_size,))

    xb = torch.stack([data[i:i + block_size] for i in ix])
    yb = torch.stack([data[i + 1:i + block_size + 1] for i in ix])

    return xb, yb

get_batch()

(tensor([[1, 1, 2, 2],
         [1, 0, 0, 2],
         [1, 0, 2, 0],
         [1, 1, 2, 2],
         [1, 2, 2, 0],
         [1, 0, 2, 0],
         [0, 1, 0, 2],
         [0, 1, 0, 0]]),
 tensor([[1, 2, 2, 0],
         [0, 0, 2, 1],
         [0, 2, 0, 1],
         [1, 2, 2, 0],
         [2, 2, 0, 1],
         [0, 2, 0, 1],
         [1, 0, 2, 0],
         [1, 0, 0, 2]]))

In [23]:
block_size = 8
batch_size = 8
n_embd = 16
n_head = 2
n_layer = 1

class MiniGPT(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, n_embd)
        self.position_embedding = nn.Embedding(block_size, n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape

        tok_emb = self.token_embedding(idx)              # [B, T, C]
        pos = torch.arange(T, device=idx.device)         # [T]
        pos_emb = self.position_embedding(pos)           # [T, C]

        x = tok_emb + pos_emb                            # [B, T, C]
        logits = self.lm_head(x)                         # [B, T, vocab_size]

        loss = None
        if targets is not None:
            B, T, C = logits.shape
            loss = F.cross_entropy(
                logits.view(B * T, C),
                targets.view(B * T),
            )

        return logits, loss


In [25]:
model = MiniGPT().to(device)

xb, yb = get_batch()
xb = xb.to(device)
yb = yb.to(device)

logits, loss = model(xb, yb)

print(xb.shape)      # [8, 8]
print(yb.shape)      # [8, 8]
print(logits.shape)  # [8, 8, 3]
print(loss.item())

torch.Size([8, 8])
torch.Size([8, 8])
torch.Size([8, 8, 3])
1.2213497161865234


## Step 7b - Add One Causal Self-Attention Head

Causal self-attention lets each token position mix information from previous positions, while preventing it from looking at future positions.

For a sequence:

```text
a b a c
```

allowed visibility is:

```text
position 0 sees: a
position 1 sees: a b
position 2 sees: a b a
position 3 sees: a b a c
```

The attention head learns three projections from the same input `x`:

```text
query: what is this position looking for?
key:   what does each position contain?
value: what information should be copied forward?
```

The query-key product computes attention scores between all pairs of positions:

```python
wei = q @ k.transpose(-2, -1)
```

Shape:

```text
[B, T, head_size] @ [B, head_size, T] -> [B, T, T]
```

The scale factor:

```python
wei = wei * (head_size ** -0.5)
```

keeps the dot-product magnitudes more stable as `head_size` changes.

The lower-triangular mask is:

```text
1 0 0 0
1 1 0 0
1 1 1 0
1 1 1 1
```

Zeros are replaced with `-inf` before softmax:

```python
wei = wei.masked_fill(self.tril[:T, :T] == 0, float("-inf"))
```

After softmax, future positions receive probability zero. The final output is a weighted mixture of value vectors:

```python
out = wei @ v
```

Shape:

```text
[B, T, T] @ [B, T, head_size] -> [B, T, head_size]
```

`register_buffer()` stores the causal mask as part of the module state without making it a trainable parameter. When the model moves to GPU, the buffer moves with it.


In [28]:
class Head(nn.Module):
      def __init__(self, head_size):
          super().__init__()
          self.key = nn.Linear(n_embd, head_size, bias=False)
          self.query = nn.Linear(n_embd, head_size, bias=False)
          self.value = nn.Linear(n_embd, head_size, bias=False)

          self.register_buffer(
              "tril",
              torch.tril(torch.ones(block_size, block_size))
          )

      def forward(self, x):
          B, T, C = x.shape

          k = self.key(x)      # [B, T, head_size]
          q = self.query(x)    # [B, T, head_size]

          wei = q @ k.transpose(-2, -1)   # [B, T, T]
          wei = wei * (k.shape[-1] ** -0.5)

          wei = wei.masked_fill(
              self.tril[:T, :T] == 0,
              float("-inf")
          )

          wei = F.softmax(wei, dim=-1)    # [B, T, T]

          v = self.value(x)               # [B, T, head_size]
          out = wei @ v                   # [B, T, head_size]

          return out

In [29]:
block_size = 8
batch_size = 8
n_embd = 16
n_head = 2
n_layer = 1

class MiniGPT(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, n_embd)
        self.position_embedding = nn.Embedding(block_size, n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)
        self.sa = Head(n_embd)

    def forward(self, idx, targets=None):
        B, T = idx.shape

        tok_emb = self.token_embedding(idx)              # [B, T, C]
        pos = torch.arange(T, device=idx.device)         # [T]
        pos_emb = self.position_embedding(pos)           # [T, C]

        x = tok_emb + pos_emb                            # [B, T, C]
        x = self.sa(x)
        logits = self.lm_head(x)                         # [B, T, vocab_size]

        loss = None
        if targets is not None:
            B, T, C = logits.shape
            loss = F.cross_entropy(
                logits.view(B * T, C),
                targets.view(B * T),
            )

        return logits, loss


In [31]:
model = MiniGPT().to(device)

xb, yb = get_batch()
xb = xb.to(device)
yb = yb.to(device)

logits, loss = model(xb, yb)

print(logits.shape)  # [batch_size, block_size, vocab_size]
print(loss.item())
print(model.sa.tril[:4, :4])

torch.Size([8, 8, 3])
1.2345659732818604
tensor([[1., 0., 0., 0.],
        [1., 1., 0., 0.],
        [1., 1., 1., 0.],
        [1., 1., 1., 1.]], device='cuda:0')


## Later Steps - Compare, Overfit, Then Vary One Thing

After single-head causal attention works, useful next checks are:

```text
frequency baseline
embedding + linear baseline
MiniGPT with causal attention
```

On this tiny dataset, the goal is not benchmark performance. The goal is to verify that each model can learn the simple next-token structure and to understand what each added component changes.

A good debugging target is intentional overfitting: use the tiny fixed corpus and train until the loss becomes very low. If the model cannot memorize a tiny dataset, something is probably wrong in the data pipeline, masking, loss shape, or optimization loop.

After that, vary one thing at a time:

```text
block_size
corpus length
embedding size
sampling temperature
number of heads/layers
```

Changing one variable at a time makes failures easier to attribute.


## Current Implementation Status

Implemented through this point:

```text
PyTorch/GPU verification
3-token character vocabulary
encoded training data
baseline context-target batching
embedding + linear baseline
sampling-based generation
GPT-style sequence targets
token + position embeddings
single causal self-attention head
MiniGPT forward pass using that attention head
```

Not yet implemented:

```text
multi-head attention
feed-forward block
residual connections
layer normalization
stacked transformer blocks
full GPT training loop after attention
```
